In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report


In [2]:
# 1. Load data
# -----------------------
df = pd.read_csv("Dataset_14-day_AA_depression_symptoms_mood_and_PHQ-9.csv")
df.columns = df.columns.str.strip()

In [3]:
# 3. KEEP ONLY IMPORTANT COLUMNS
# -----------------------------------
keep_cols = [
    "user_id", "time",
    "phq1","phq2","phq3","phq4","phq5","phq6","phq7","phq8","phq9",
    "happiness.score"
]

df = df[keep_cols]

In [4]:
# 4. HANDLE MISSING VALUES
# -----------------------------------

# PHQ columns → fill with 0 (safe assumption)
phq_cols = ["phq1","phq2","phq3","phq4","phq5","phq6","phq7","phq8","phq9"]
df[phq_cols] = df[phq_cols].fillna(0)

# happiness score → fill with mean
df["happiness.score"] = df["happiness.score"].fillna(df["happiness.score"].mean())

In [5]:

# -----------------------------------
# 5. CONVERT TIME
# -----------------------------------
df["time"] = pd.to_datetime(df["time"])

# -----------------------------------
# 6. CREATE PHQ SCORE
# -----------------------------------
df["phq_score"] = df[phq_cols].sum(axis=1)


In [6]:
# 7. CREATE LABEL (TARGET)
# -----------------------------------
def label(score):
    if score <= 4:
        return 0   # Minimal
    elif score <= 9:
        return 1   # Mild
    elif score <= 14:
        return 2   # Moderate
    else:
        return 3   # Severe

df["label"] = df["phq_score"].apply(label)

In [7]:
# 8. SORT FOR TIME SERIES
# -----------------------------------
df = df.sort_values(["user_id", "time"])

# -----------------------------------
# 9. CREATE ROLLING AVERAGE
# -----------------------------------
df["rolling_3"] = (
    df.groupby("user_id")["phq_score"]
    .rolling(window=3)
    .mean()
    .reset_index(0, drop=True)
)

# fill initial NaN
df["rolling_3"] = df["rolling_3"].fillna(df["phq_score"])

In [9]:
# 10. FINAL CHECK
# -----------------------------------
print(df.head(55))
print(df.shape)

     user_id                time  phq1  phq2  phq3  phq4  phq5  phq6  phq7  \
43         1 2017-01-09 07:22:37   3.0   3.0   3.0   3.0   2.0   3.0   1.0   
42         1 2017-01-13 18:58:46   3.0   3.0   3.0   3.0   2.0   3.0   1.0   
19         1 2017-01-14 15:11:14   3.0   3.0   3.0   3.0   2.0   3.0   1.0   
41         1 2017-01-14 21:26:27   3.0   3.0   3.0   3.0   2.0   3.0   1.0   
33         1 2017-01-15 12:06:22   3.0   3.0   3.0   3.0   2.0   3.0   1.0   
31         1 2017-01-15 23:25:55   3.0   3.0   3.0   3.0   2.0   3.0   1.0   
22         1 2017-01-16 12:37:56   3.0   3.0   3.0   3.0   2.0   3.0   1.0   
32         1 2017-01-16 20:14:09   3.0   3.0   3.0   3.0   2.0   3.0   1.0   
13         1 2017-01-17 07:21:29   3.0   3.0   3.0   3.0   2.0   3.0   1.0   
21         1 2017-01-17 23:33:05   3.0   3.0   3.0   3.0   2.0   3.0   1.0   
14         1 2017-01-18 15:01:07   3.0   3.0   3.0   3.0   2.0   3.0   1.0   
15         1 2017-01-18 18:01:18   3.0   3.0   3.0   3.0   2.0  